In [1]:
!pip install datasets
!pip install datasets transformers torch accelerate openai nltk tqdm

In [2]:
import torch
import time
import openai
import nltk
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm
from scipy.spatial.distance import cosine
from PIL import Image, ExifTags

In [1]:
from datasets import load_dataset

# Load the specific subset of MMLU
dataset = load_dataset("tasksource/mmlu", "professional_law")


# Display the dataset structure
print(dataset)

# Show the length (number of rows) for each split
for split in dataset.keys():
    print(f"Number of rows in {split}: {len(dataset[split])}")

# Show column names
for split in dataset.keys():
    print(f"Columns in {split}: {dataset[split].column_names}")


0000.parquet:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/115k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/14.7k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/1534 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/170 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['question', 'choices', 'answer'],
        num_rows: 1534
    })
    validation: Dataset({
        features: ['question', 'choices', 'answer'],
        num_rows: 170
    })
    dev: Dataset({
        features: ['question', 'choices', 'answer'],
        num_rows: 5
    })
})
Number of rows in test: 1534
Number of rows in validation: 170
Number of rows in dev: 5
Columns in test: ['question', 'choices', 'answer']
Columns in validation: ['question', 'choices', 'answer']
Columns in dev: ['question', 'choices', 'answer']


In [2]:
# Convert to a Pandas DataFrame
import pandas as pd

df = pd.DataFrame(dataset['test'])  # Assuming "test" is the main split

# Check the number of rows
print(f"Number of rows: {len(df)}")

# Display first few rows
df.head()


Number of rows: 1534


,question,choices,answer
0,"One afternoon, a pilot was flying a small airp...","[admissible, because the attorney-client privi...",2
1,"A state statute provides: ""Whenever a person k...","[not guilty, if the arrest was unlawful withou...",1
2,A taxpayer was notified by the government that...,"[inadmissible, because it would violate the at...",3
3,A resident announced his candidacy for state r...,[The resident's petition contained a large num...,2
4,A defendant was angry at his friend for marryi...,[He intended to kill the friend and not the da...,1


In [12]:
# Save as CSV
df.to_csv("professional_medicine_mmlu.csv", index=False)


## Load Llama 3.1 7b

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import os
from huggingface_hub import login

In [6]:
# Load Llama 3.1 7B Instruct model and tokenizer
model_name = "meta-llama/Llama-3.1-8B-Instruct"
hf_token = os.getenv("HF_TOKEN")

tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    token=hf_token
)

# Verify Model Loading
print("Model and tokenizer loaded successfully!")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model and tokenizer loaded successfully!


In [7]:
def get_llama_answer(question, choices):
    """Function to get the correct answer's index using Llama-3.1-8B"""
    formatted_prompt = f"Question: {question}\nChoices: {choices}\nChoose the correct answer and return its order number."
    
    # Tokenize input
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")  # Move to GPU if available
    
    # Generate response
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=50)
    
    # Decode and extract the predicted answer
    response_text = tokenizer.decode(output[0], skip_special_tokens=True)
    
    # Extract the correct answer from the response
    for idx, choice in enumerate(choices, 1):  # Start index at 1
        if choice in response_text:
            return idx
    
    return None  # In case no valid choice is found

# Apply the function to each row
df["LLAMA_Answer"] = df.apply(lambda row: get_llama_answer(row["question"], row["choices"]), axis=1)

# Display results
print(df)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

...

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

                                               question  \
0     One afternoon, a pilot was flying a small airp...   
1     A state statute provides: "Whenever a person k...   
2     A taxpayer was notified by the government that...   
3     A resident announced his candidacy for state r...   
4     A defendant was angry at his friend for marryi...   
...                                                 ...   
1529  A man and woman lived together but were never ...   
1530  During a deer-hunting season open to rifle hun...   
1531  A man was charged with tax fraud. He wished to...   
1532  A victim was standing on a street corner waiti...   
1533  A homeowner said to a roofer, "My roof leaks. ...   

                                                choices  answer  LLAMA_Answer  
0     [admissible, because the attorney-client privi...       2             1  
1     [not guilty, if the arrest was unlawful withou...       1             1  
2     [inadmissible, because it would violate the a

In [9]:
# Save df_subset to a CSV file
df.to_csv("llama_answer_law_direct.csv", index=False)

print("CSV file saved as 'llama_answer_law_direct.csv'")

CSV file saved as 'llama_answer_law_direct.csv'


In [8]:
# Compute accuracy
if "LLAMA_Answer" in df.columns and "answer" in df.columns:
    total_predictions = len(df)
    correct_predictions = (df["LLAMA_Answer"] == df["answer"]).sum()
    accuracy = (correct_predictions / total_predictions) * 100

    # Display results
    print(f"Total Predictions: {total_predictions}")
    print(f"Correct Predictions: {correct_predictions}")
    print(f"Accuracy: {accuracy:.2f}%")
else:
    print("Error: Ensure 'LLAMA_Answer' and 'answer' exist in df.")


Total Predictions: 1534
Correct Predictions: 368
Accuracy: 23.99%


In [ ]:
# Initialize extracted values
    predicted_aid = None
    predicted_atext = None
    reasoning_steps = []

    # Extract prediction and reasoning
    for line in response_text.split("\n"):
        line = line.strip()
        
        # Extract the predicted answer
        if line.startswith("Predicted Correct Answer:") or line.startswith("Final Decision:"):
            parts = line.split(":")
            if len(parts) > 1:
                prediction = parts[1].strip().split(" - ")
                if len(prediction) == 2:
                    predicted_aid, predicted_atext = prediction
        
        # Extract reasoning steps
        elif line.startswith("Step"):
            reasoning_steps.append(line)

    # Join reasoning steps for structured output
    reasoning_text = " | ".join(reasoning_steps)

    return predicted_aid, predicted_atext, reasoning_text, end_time - start_time

## From 1) Clinical Evidence, 2) Relavancy, 3) Logistic Model

In [16]:
def generate_llama_response(prompt):
    """Generates a response using Llama-3.1-8B-Instruct"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")  # Move to GPU if available
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=300)
    response = tokenizer.decode(output[0], skip_special_tokens=True).strip()
    return response

# Function to extract clinical evidence using Llama
def extract_clinical_evidence(text):
    prompt = f"""
    Extract key clinical evidence from the following medical question. 
    Return a list of extracted phrases that indicate medical findings, treatments, symptoms, or clinical concepts.

    Question: {text}
    """
    
    response_text = generate_llama_response(prompt)
    
    # Ensure response is formatted as a list of extracted phrases
    evidence_list = [e.strip() for e in response_text.split("\n") if e.strip()]

    if not evidence_list:
        print(f"Warning: No evidence extracted from Llama response: {response_text}")
        return []

    return evidence_list

In [20]:
import pandas as pd

# Process only the first 5 rows of df
df_subset = df.head(5).copy()  # Ensure we work on a copy to avoid modifying the original df

# Extract clinical evidence using Llama for each question and store in a new column
df_subset["LLAMA_Evidence"] = df_subset["question"].apply(extract_clinical_evidence)

# Display the updated DataFrame
print(df_subset)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


                                            question  \
0  A 67-year-old woman comes to the physician for...   
1  A 25-year-old gravida 3 para 2 female is admit...   
2  A 5-year-old boy is brought to the physician b...   
3  A 9-year-old boy is brought to the office by h...   
4  A 25-year-old woman comes to the physician bec...   

                                             choices  answer  LLAMA_Answer  \
0  [Cerebral infarction during the hospitalizatio...       2           1.0   
1  [Braxton Hicks contractions, lower uterine ret...       2           1.0   
2  [The findings are clinically and statistically...       1           1.0   
3  [Atrial fibrillation, Cor pulmonale, Systemic ...       2           1.0   
4  [Median nerve at the wrist, Musculocutaneous n...       3           1.0   

                                      LLAMA_Evidence  
0  [Extract key clinical evidence from the follow...  
1  [Extract key clinical evidence from the follow...  
2  [Extract key clinical evid

In [21]:
# Save df_subset to a CSV file
df_subset.to_csv("llama_evidence_extraction.csv", index=False)

print("CSV file saved as 'llama_evidence_extraction.csv'")


CSV file saved as 'llama_evidence_extraction.csv'


In [ ]:
# Function to assign relevance scores using Llama
def assign_relevance_scores(evidence_list, question):
    scores = []
    for evidence in evidence_list:
        prompt = f"""
        Consider the following clinical evidence: "{evidence}"
        in the context of the medical question: "{question}".
        Assign a relevance score from 0-3 (0: Not relevant, 3: Highly relevant).
        Only return the score (0, 1, 2, or 3).
        """

        response_text = generate_llama_response(prompt).strip()

        try:
            score = int(response_text)  # Convert response to an integer
            if score in [0, 1, 2, 3]:  # Ensure it's within expected range
                scores.append(score)
            else:
                print(f"Warning: Invalid score '{response_text}' from Llama. Assigning default 0.")
                scores.append(0)  # Assign a default score if invalid
        except ValueError:
            print(f"Warning: Unexpected Llama response '{response_text}'. Assigning default 0.")
            scores.append(0)  # Assign a default score if parsing fails

    return scores

In [19]:
from IPython.core.display import display, HTML

def extract_clinical_evidence(text):
    prompt = f"""
    Extract key clinical evidence from the following medical question. 
    Highlight phrases that indicate medical findings, treatments, symptoms, or clinical concepts.

    Question: {text}

    Return the extracted phrases as a list.
    """

    response_text = generate_llama_response(prompt)

    # Format extracted evidence as a list
    evidence_list = [e.strip() for e in response_text.split("\n") if e.strip()]

    if not evidence_list:
        print(f"Warning: No evidence extracted from Llama response: {response_text}")
        return []

    return evidence_list


def assign_relevance_scores(evidence_list, question):
    scores = []
    for evidence in evidence_list:
        prompt = f"""
        Consider the following clinical evidence: "{evidence}"
        in the context of the medical question: "{question}".
        Assign a relevance score from 1-3:
        - 1 = Irrelevant (not useful for diagnosis)
        - 2 = Low relevance (partially useful)
        - 3 = High relevance (important for diagnosis)

        Only return the score (1, 2, or 3).
        """

        response_text = generate_llama_response(prompt).strip()

        try:
            score = int(response_text)  
            if score in [1, 2, 3]:  
                scores.append(score)
            else:
                print(f"Warning: Invalid score '{response_text}' from Llama. Assigning default 1.")
                scores.append(1)  
        except ValueError:
            print(f"Warning: Unexpected Llama response '{response_text}'. Assigning default 1.")
            scores.append(1)

    return scores


def map_color_coding(scores):
    color_mapping = {1: "red", 2: "yellow", 3: "green"}
    return [color_mapping[score] for score in scores]


import pandas as pd

# Process only the first 5 rows of df for testing
df_subset = df.head(5)
data = []

for _, row in df_subset.iterrows():
    question = row["question"]
    
    # Extract clinical evidence
    evidence_list = extract_clinical_evidence(question)

    # Assign relevance scores
    relevancy_scores = assign_relevance_scores(evidence_list, question)

    # Ensure lengths match before storing data
    if len(evidence_list) != len(relevancy_scores):
        print(f"Warning: Mismatch between evidence ({len(evidence_list)}) and scores ({len(relevancy_scores)}) for question: {question}")
        continue  

    # Assign colors based on relevance scores
    color_codes = map_color_coding(relevancy_scores)

    # Store extracted data
    for evidence, relevancy, color in zip(evidence_list, relevancy_scores, color_codes):
        data.append({
            "question": question,
            "evidence": evidence,
            "relevancy_score": relevancy,
            "color_coding": color
        })

# Convert to DataFrame
df_processed = pd.DataFrame(data)

# Display the processed DataFrame
print(df_processed)

def highlight_text(text, evidence_list, color_codes):
    """Highlight extracted evidence in the question with assigned colors."""
    for evidence, color in zip(evidence_list, color_codes):
        text = text.replace(evidence, f'<span style="background-color:{color};">{evidence}</span>')
    return text

# Apply color highlighting for each row
df_processed["highlighted_text"] = df_processed.apply(
    lambda row: highlight_text(row["question"], [row["evidence"]], [row["color_coding"]]), axis=1
)

# Display highlighted text
for _, row in df_processed.iterrows():
    display(HTML(f"<p>{row['highlighted_text']}</p>"))


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


        in the context of the medical question: "A 67-year-old woman comes to the physician for a follow-up examination. She had a pulmonary embolism and required treatment in the hospital for 3 weeks. She had a retroperitoneal hemorrhage; anticoagulant therapy was temporarily discontinued, and she underwent placement of an inferior vena cava (IVC) filter. She had a hematoma that was resolving on discharge from the hospital 2 weeks ago. Today, she says she has had a persistent sensation of tingling and numbness of her left thigh that she did not report in the hospital because she thought it would go away; the sensation has improved somewhat during the past week. Her only medication is warfarin. Vital signs are within normal limits. Examination of the skin shows no abnormalities. Muscle strength is normal. Sensation to light touch is decreased over a 5 x 5-cm area on the lateral aspect of the left anterior thigh. Which of the following is the most likely cause of this patient's decrease

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


        in the context of the medical question: "A 67-year-old woman comes to the physician for a follow-up examination. She had a pulmonary embolism and required treatment in the hospital for 3 weeks. She had a retroperitoneal hemorrhage; anticoagulant therapy was temporarily discontinued, and she underwent placement of an inferior vena cava (IVC) filter. She had a hematoma that was resolving on discharge from the hospital 2 weeks ago. Today, she says she has had a persistent sensation of tingling and numbness of her left thigh that she did not report in the hospital because she thought it would go away; the sensation has improved somewhat during the past week. Her only medication is warfarin. Vital signs are within normal limits. Examination of the skin shows no abnormalities. Muscle strength is normal. Sensation to light touch is decreased over a 5 x 5-cm area on the lateral aspect of the left anterior thigh. Which of the following is the most likely cause of this patient's decrease

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


        in the context of the medical question: "A 67-year-old woman comes to the physician for a follow-up examination. She had a pulmonary embolism and required treatment in the hospital for 3 weeks. She had a retroperitoneal hemorrhage; anticoagulant therapy was temporarily discontinued, and she underwent placement of an inferior vena cava (IVC) filter. She had a hematoma that was resolving on discharge from the hospital 2 weeks ago. Today, she says she has had a persistent sensation of tingling and numbness of her left thigh that she did not report in the hospital because she thought it would go away; the sensation has improved somewhat during the past week. Her only medication is warfarin. Vital signs are within normal limits. Examination of the skin shows no abnormalities. Muscle strength is normal. Sensation to light touch is decreased over a 5 x 5-cm area on the lateral aspect of the left anterior thigh. Which of the following is the most likely cause of this patient's decrease

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


        in the context of the medical question: "A 67-year-old woman comes to the physician for a follow-up examination. She had a pulmonary embolism and required treatment in the hospital for 3 weeks. She had a retroperitoneal hemorrhage; anticoagulant therapy was temporarily discontinued, and she underwent placement of an inferior vena cava (IVC) filter. She had a hematoma that was resolving on discharge from the hospital 2 weeks ago. Today, she says she has had a persistent sensation of tingling and numbness of her left thigh that she did not report in the hospital because she thought it would go away; the sensation has improved somewhat during the past week. Her only medication is warfarin. Vital signs are within normal limits. Examination of the skin shows no abnormalities. Muscle strength is normal. Sensation to light touch is decreased over a 5 x 5-cm area on the lateral aspect of the left anterior thigh. Which of the following is the most likely cause of this patient's decrease

KeyboardInterrupt: 